# 05 · NER Dataset Validation

Comprehensive validation of `integrated_conversations_ner.csv` produced by notebook `04_scispacy_ner.ipynb`.

**Goals:**
1. Audit raw entity counts, label distribution, and coverage
2. Detect and remove **duplicate utterances** (exact & near-duplicate at dialogue level)
3. Detect and remove **misnamed / noisy entities** (false positives, stop-words, numeric-only tokens, overly short tokens, wrong-label assignments)
4. Produce a **full validation report** (summary table + per-category breakdowns)
5. Save a clean, validated CSV for downstream use

**Expected entity labels:** `DISEASE`, `CHEMICAL` (from `en_ner_bc5cdr_md`)

---
## 0 · Imports & Configuration

In [ ]:
import re
import ast
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.float_format', '{:,.2f}'.format)

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR   = Path('../data/processed')
INPUT_CSV  = DATA_DIR / 'integrated_conversations_ner.csv'
OUTPUT_CSV = DATA_DIR / 'integrated_conversations_ner_validated.csv'

# ── Entity label constants ─────────────────────────────────────────────────
VALID_LABELS   = {'DISEASE', 'CHEMICAL'}
PIPE_SEP       = ' | '          # delimiter used by notebook 04

print('Input :', INPUT_CSV)
print('Output:', OUTPUT_CSV)

---
## 1 · Load Data

In [ ]:
df = pd.read_csv(
    INPUT_CSV,
    low_memory=False,
    dtype={'original_id': str, 'dialogue_id': str}
)

print(f'Rows loaded : {len(df):,}')
print(f'Columns     : {list(df.columns)}')
df.head(3)

---
## 2 · Schema & Integrity Check

In [ ]:
EXPECTED_COLS = [
    'dialogue_id', 'turn_id', 'speaker',
    'utterance', 'source_dataset', 'original_id',
    'extracted_entities', 'entity_labels'
]

missing_cols = [c for c in EXPECTED_COLS if c not in df.columns]
extra_cols   = [c for c in df.columns    if c not in EXPECTED_COLS]

print('=== Schema Check ===')
print(f'  Expected columns present : {len(EXPECTED_COLS) - len(missing_cols)}/{len(EXPECTED_COLS)}')
if missing_cols:
    print(f'  ⚠ MISSING columns       : {missing_cols}')
if extra_cols:
    print(f'  Extra columns found      : {extra_cols}')

print('\n=== Null / Empty Counts ===')
null_summary = df.isnull().sum().rename('null_count')
empty_summary = (df == '').sum().rename('empty_string_count')
display(pd.concat([null_summary, empty_summary], axis=1))

In [ ]:
# Normalise entity columns: NaN → empty string
df['extracted_entities'] = df['extracted_entities'].fillna('')
df['entity_labels']      = df['entity_labels'].fillna('')

# Normalise utterance
df['utterance'] = df['utterance'].fillna('').astype(str).str.strip()

print('Null values after normalisation:')
print(df[['utterance','extracted_entities','entity_labels']].isnull().sum())

---
## 3 · Parse Entity Columns

In [ ]:
def split_pipe(value: str) -> list:
    """Split a pipe-delimited entity string into a clean list."""
    if not value or not isinstance(value, str) or value.strip() == '':
        return []
    return [tok.strip() for tok in value.split('|') if tok.strip() != '']

df['ents_list']   = df['extracted_entities'].apply(split_pipe)
df['labels_list'] = df['entity_labels'].apply(split_pipe)

# ── Alignment check: same number of entities and labels per row ────────────
df['ent_count']   = df['ents_list'].apply(len)
df['label_count'] = df['labels_list'].apply(len)
misaligned = df[df['ent_count'] != df['label_count']]

print(f'Total rows              : {len(df):,}')
print(f'Rows with entities      : {(df["ent_count"] > 0).sum():,}')
print(f'Rows without entities   : {(df["ent_count"] == 0).sum():,}')
print(f'Misaligned rows         : {len(misaligned):,}  ← should be 0')
if len(misaligned) > 0:
    display(misaligned[['dialogue_id','utterance','extracted_entities','entity_labels']].head(5))

---
## 4 · Label Distribution (raw)

In [ ]:
# Explode into a flat (entity_text, label) table
pairs = [
    (ent, lbl)
    for ents, lbls in zip(df['ents_list'], df['labels_list'])
    for ent, lbl in zip(ents, lbls)
]

ent_df = pd.DataFrame(pairs, columns=['entity_text', 'label'])
ent_df['entity_lower'] = ent_df['entity_text'].str.lower().str.strip()

print(f'Total entity mentions (raw) : {len(ent_df):,}')
print('\n--- Label Breakdown ---')
label_counts = ent_df['label'].value_counts(dropna=False)
display(label_counts.to_frame('count').assign(
    pct=lambda x: (x['count'] / len(ent_df) * 100).round(2)
))

# Flag unexpected labels
unexpected = ent_df[~ent_df['label'].isin(VALID_LABELS)]
print(f'\n⚠ Unexpected labels: {len(unexpected):,}')
if len(unexpected) > 0:
    display(unexpected['label'].value_counts())

---
## 5 · Detect & Remove Duplicate Utterances

In [ ]:
# ── 5a. Exact duplicate utterances ────────────────────────────────────────
df['utt_norm'] = df['utterance'].str.lower().str.strip()

exact_dup_mask = df.duplicated(subset=['utt_norm'], keep='first') & (df['utt_norm'] != '')
n_exact_dups = exact_dup_mask.sum()

print(f'=== Duplicate Utterance Analysis ===')
print(f'Total rows                  : {len(df):,}')
print(f'Exact duplicate utterances  : {n_exact_dups:,}')

# Show top repeated utterances
top_repeated = (
    df[df['utt_norm'] != '']
    .groupby('utt_norm')
    .size()
    .sort_values(ascending=False)
    .head(20)
    .rename('occurrence_count')
    .reset_index()
)
top_repeated.columns = ['utterance_normalised', 'occurrence_count']
multi = top_repeated[top_repeated['occurrence_count'] > 1]
print(f'\nTop repeated utterances (sample):')
display(multi.head(15))

In [ ]:
# ── 5b. Duplicate (dialogue_id, turn_id) rows ─────────────────────────────
turn_dup_mask = df.duplicated(subset=['dialogue_id', 'turn_id'], keep='first')
n_turn_dups   = turn_dup_mask.sum()
print(f'Duplicate (dialogue_id, turn_id) pairs : {n_turn_dups:,}')
if n_turn_dups > 0:
    display(df[turn_dup_mask][['dialogue_id','turn_id','speaker','utterance']].head(5))

In [ ]:
# ── 5c. Remove duplicates ─────────────────────────────────────────────────
# Strategy: drop exact utterance duplicates UNLESS they occur in DIFFERENT
# dialogues (same phrase OK across dialogues, but not within the same dialogue).
# Also drop structural (dialogue_id, turn_id) duplicates.

n_before = len(df)

# 1) Drop duplicate (dialogue_id, turn_id) pairs
df = df.drop_duplicates(subset=['dialogue_id', 'turn_id'], keep='first')

# 2) Drop exact same utterance within the SAME dialogue
df = df.drop_duplicates(subset=['dialogue_id', 'utt_norm'], keep='first')

n_after  = len(df)
n_removed = n_before - n_after

print(f'Rows before deduplication : {n_before:,}')
print(f'Rows after  deduplication : {n_after:,}')
print(f'Rows removed              : {n_removed:,}  ({n_removed/n_before*100:.2f}%)')
df = df.reset_index(drop=True)

---
## 6 · Detect & Remove Misnamed / Noisy Entities

The following heuristic rules flag entities as noisy:

| Rule | Rationale |
|---|---|
| Token is numeric-only | Numbers like `"2"`, `"10mg"` are not clinical entities |
| Token length < 2 characters | Single-char tokens (e.g. `"a"`, `"I"`) are stop-words |
| Token is a common English stop-word | Words like `"hi"`, `"no"`, `"yes"`, `"the"` |
| Token contains only punctuation/whitespace | Parsing artefacts |
| Token is a pure number or percentage | `"100%"`, `"3.5"` |
| CHEMICAL label on a non-chemical word | Heuristic blacklist of common false positives |
| Token is all uppercase and < 3 chars | Abbreviation artefacts like `"Al"`, `"IV"` unless confirmed |
| Entity text has a label not in VALID_LABELS | Unknown labels |

In [ ]:
import string

# ── Stop-word list (common English words that slip through NER) ────────────
STOPWORDS = {
    'a','an','the','and','or','but','in','on','at','to','for','of','with',
    'by','from','as','is','was','are','were','be','been','being','have',
    'has','had','do','does','did','will','would','could','should','may',
    'might','shall','can','need','dare','ought','used','it','its','he',
    'she','they','we','i','you','hi','hello','ok','okay','yes','no','not',
    'also','just','so','this','that','these','those','there','here','all',
    'any','both','each','few','more','most','other','some','such','then',
    'than','too','very','s','t','don','doesn','didn','isn','wasn','aren',
    'weren','wouldn','couldn','shouldn','won','can\'t','whats','what',
    'how','when','where','why','who','which','because','if','about','after',
    'before','between','into','through','during','without','within','against',
    'am','me','my','our','your','their','his','her','its','well','now',
    'only','even','still','since','while','although','however',
}

# ── Confirmed false-positive tokens for CHEMICAL label ────────────────────
CHEMICAL_FP = {
    'hi', 'hello', 'al', 'fe', 'na', 'k', 'ca', 'mg',  # element symbols misused
    'oil', 'air', 'gas', 'water', 'salt',                # too generic
    'no', 'yes', 'ok', 'so', 'or',                       # function words
    'a', 'an', 'the',
}

# ── Regex patterns ────────────────────────────────────────────────────────
_RE_NUMERIC    = re.compile(r'^[\d\s,\.\-\+%/]+$')     # pure numbers / percentages
_RE_PUNCT_ONLY = re.compile(r'^[\W_]+$')                # punctuation / whitespace only
_RE_DOSE_UNIT  = re.compile(                            # dose strings like "10mg", "2x"
    r'^\d+[\.,]?\d*\s*(mg|mcg|ug|ml|mmol|g|kg|iu|unit|units|x|%|tablet|tablets|cap|capsule)s?$',
    re.IGNORECASE
)

def classify_entity(text: str, label: str) -> str:
    """
    Returns 'valid' or one of several rejection reasons.
    """
    if not isinstance(text, str) or text.strip() == '':
        return 'empty'

    t = text.strip()
    t_lower = t.lower()

    # Unknown label
    if label not in VALID_LABELS:
        return 'unknown_label'

    # Punctuation/whitespace only
    if _RE_PUNCT_ONLY.match(t):
        return 'punct_only'

    # Too short
    if len(t) < 2:
        return 'too_short'

    # Numeric only or dose string
    if _RE_NUMERIC.match(t):
        return 'numeric_only'
    if _RE_DOSE_UNIT.match(t):
        return 'dose_unit'

    # Stop-word
    if t_lower in STOPWORDS:
        return 'stopword'

    # CHEMICAL false positive
    if label == 'CHEMICAL' and t_lower in CHEMICAL_FP:
        return 'chemical_fp'

    # Single uppercase abbreviation (2 chars) — likely not a real entity
    if len(t) == 2 and t.isupper() and label == 'CHEMICAL':
        return 'short_uppercase_chemical'

    return 'valid'

print('classify_entity() loaded.')

In [ ]:
# ── Apply classification to each entity in the exploded table ─────────────
ent_df2 = pd.DataFrame(
    [
        (i, ent, lbl, classify_entity(ent, lbl))
        for i, (ents, lbls) in enumerate(zip(df['ents_list'], df['labels_list']))
        for ent, lbl in zip(ents, lbls)
    ],
    columns=['row_idx', 'entity_text', 'label', 'verdict']
)

ent_df2['entity_lower'] = ent_df2['entity_text'].str.lower().str.strip()

# ── Summary ───────────────────────────────────────────────────────────────
print(f'Total entity mentions parsed : {len(ent_df2):,}')
verdict_summary = ent_df2['verdict'].value_counts().rename('count').to_frame()
verdict_summary['pct'] = (verdict_summary['count'] / len(ent_df2) * 100).round(3)
display(verdict_summary)

In [ ]:
# ── Show samples of each rejection category ────────────────────────────────
REJECTION_CATEGORIES = [v for v in ent_df2['verdict'].unique() if v != 'valid']

for cat in REJECTION_CATEGORIES:
    sub = ent_df2[ent_df2['verdict'] == cat]
    top = sub['entity_text'].value_counts().head(10)
    print(f"\n{'='*55}")
    print(f"Rejection category: '{cat}'  ({len(sub):,} mentions)")
    print(f"{'='*55}")
    display(top.to_frame('count'))

In [ ]:
# ── Build per-row clean entity & label lists ──────────────────────────────
valid_by_row = (
    ent_df2[ent_df2['verdict'] == 'valid']
    .groupby('row_idx')
    .apply(lambda g: list(zip(g['entity_text'], g['label'])))
)

# Reconstruct clean lists aligned with df index
def build_clean_lists(df):
    clean_ents   = []
    clean_labels = []
    for i in range(len(df)):
        if i in valid_by_row.index:
            pairs = valid_by_row[i]
            clean_ents.append([p[0] for p in pairs])
            clean_labels.append([p[1] for p in pairs])
        else:
            clean_ents.append([])
            clean_labels.append([])
    return clean_ents, clean_labels

df['clean_entities'], df['clean_labels'] = build_clean_lists(df)

# Serialise back to pipe-delimited strings (same format as input)
df['extracted_entities_clean'] = df['clean_entities'].apply(
    lambda lst: ' | '.join(lst) if lst else ''
)
df['entity_labels_clean'] = df['clean_labels'].apply(
    lambda lst: ' | '.join(lst) if lst else ''
)

# Entity counts before / after cleaning
df['ent_count_clean'] = df['clean_entities'].apply(len)

total_before = df['ent_count'].sum()
total_after  = df['ent_count_clean'].sum()
print(f'Entity mentions before cleaning : {total_before:,}')
print(f'Entity mentions after  cleaning : {total_after:,}')
print(f'Entities removed                : {total_before - total_after:,}  '
      f'({(total_before - total_after)/total_before*100:.2f}%)')

---
## 7 · Duplicate Entity Mentions Within a Row

In [ ]:
def dedup_entities(ents: list, lbls: list) -> tuple:
    """
    Remove consecutive / exact duplicate (entity, label) pairs within a single row.
    Preserves first occurrence.
    """
    seen = set()
    out_e, out_l = [], []
    for e, l in zip(ents, lbls):
        key = (e.lower().strip(), l)
        if key not in seen:
            seen.add(key)
            out_e.append(e)
            out_l.append(l)
    return out_e, out_l

results = df.apply(
    lambda r: dedup_entities(r['clean_entities'], r['clean_labels']),
    axis=1
)
df['clean_entities'] = results.apply(lambda x: x[0])
df['clean_labels']   = results.apply(lambda x: x[1])

df['extracted_entities_clean'] = df['clean_entities'].apply(
    lambda lst: ' | '.join(lst) if lst else ''
)
df['entity_labels_clean'] = df['clean_labels'].apply(
    lambda lst: ' | '.join(lst) if lst else ''
)

df['ent_count_clean'] = df['clean_entities'].apply(len)

intra_removed = total_after - df['ent_count_clean'].sum()
print(f'Intra-row duplicate entity pairs removed : {intra_removed:,}')
print(f'Final entity mention count               : {df["ent_count_clean"].sum():,}')

---
## 8 · Full Validation Report

In [ ]:
# ── Re-build clean entity flat table for stats ────────────────────────────
clean_pairs = [
    (ent, lbl)
    for ents, lbls in zip(df['clean_entities'], df['clean_labels'])
    for ent, lbl in zip(ents, lbls)
]
clean_ent_df = pd.DataFrame(clean_pairs, columns=['entity_text', 'label'])
clean_ent_df['entity_lower'] = clean_ent_df['entity_text'].str.lower().str.strip()

# ── Section 8.1 : Row-level summary ───────────────────────────────────────
display(HTML('<h3>8.1 · Row-level Summary</h3>'))
row_summary = pd.DataFrame({
    'Metric': [
        'Total rows (raw input)',
        'Rows after deduplication',
        'Rows removed (duplicates)',
        '% rows removed',
        'Rows with ≥1 entity (clean)',
        'Rows with 0 entities (clean)',
        'Unique dialogues',
        'Unique speakers',
        'Source datasets',
    ],
    'Value': [
        f"{n_before:,}",
        f"{len(df):,}",
        f"{n_before - len(df):,}",
        f"{(n_before - len(df))/n_before*100:.2f}%",
        f"{(df['ent_count_clean'] > 0).sum():,}",
        f"{(df['ent_count_clean'] == 0).sum():,}",
        f"{df['dialogue_id'].nunique():,}",
        f"{df['speaker'].nunique()}",
        ", ".join(df['source_dataset'].unique()),
    ]
})
display(row_summary)

In [ ]:
# ── Section 8.2 : Entity-level summary ────────────────────────────────────
display(HTML('<h3>8.2 · Entity-level Summary</h3>'))

n_raw_mentions    = len(ent_df2)
n_clean_mentions  = len(clean_ent_df)
n_noisy_mentions  = n_raw_mentions - n_clean_mentions

entity_summary = pd.DataFrame({
    'Metric': [
        'Total raw entity mentions',
        'Valid entity mentions (after cleaning)',
        'Noisy / removed entity mentions',
        '% entities removed',
        'Unique valid entity texts',
        'DISEASE mentions (clean)',
        'CHEMICAL mentions (clean)',
        'Rows with 0 entities after cleaning',
    ],
    'Value': [
        f"{n_raw_mentions:,}",
        f"{n_clean_mentions:,}",
        f"{n_noisy_mentions:,}",
        f"{n_noisy_mentions/n_raw_mentions*100:.2f}%",
        f"{clean_ent_df['entity_lower'].nunique():,}",
        f"{(clean_ent_df['label']=='DISEASE').sum():,}",
        f"{(clean_ent_df['label']=='CHEMICAL').sum():,}",
        f"{(df['ent_count_clean']==0).sum():,}",
    ]
})
display(entity_summary)

In [ ]:
# ── Section 8.3 : Noisy entity breakdown by rejection reason ──────────────
display(HTML('<h3>8.3 · Noisy Entity Breakdown by Rejection Reason</h3>'))

rejection_df = (
    ent_df2[ent_df2['verdict'] != 'valid']
    .groupby(['verdict', 'label'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
display(rejection_df)

In [ ]:
# ── Section 8.4 : Top 30 DISEASE entities (clean) ─────────────────────────
display(HTML('<h3>8.4 · Top 30 DISEASE Entities (clean)</h3>'))

top_disease = (
    clean_ent_df[clean_ent_df['label'] == 'DISEASE']['entity_lower']
    .value_counts()
    .head(30)
    .reset_index()
)
top_disease.columns = ['entity', 'count']
display(top_disease)

In [ ]:
# ── Section 8.5 : Top 30 CHEMICAL entities (clean) ────────────────────────
display(HTML('<h3>8.5 · Top 30 CHEMICAL Entities (clean)</h3>'))

top_chemical = (
    clean_ent_df[clean_ent_df['label'] == 'CHEMICAL']['entity_lower']
    .value_counts()
    .head(30)
    .reset_index()
)
top_chemical.columns = ['entity', 'count']
display(top_chemical)

In [ ]:
# ── Section 8.6 : Source dataset breakdown (clean) ────────────────────────
display(HTML('<h3>8.6 · Entity Coverage by Source Dataset (clean)</h3>'))

source_stats = (
    df.groupby('source_dataset')
    .agg(
        total_rows=('utterance', 'count'),
        rows_with_entities=('ent_count_clean', lambda x: (x > 0).sum()),
        total_entity_mentions=('ent_count_clean', 'sum'),
        avg_entities_per_row=('ent_count_clean', 'mean'),
    )
    .reset_index()
)
source_stats['coverage_pct'] = (
    source_stats['rows_with_entities'] / source_stats['total_rows'] * 100
).round(2)
display(source_stats)

In [ ]:
# ── Section 8.7 : Speaker balance ─────────────────────────────────────────
display(HTML('<h3>8.7 · Entity Coverage by Speaker</h3>'))

speaker_stats = (
    df.groupby('speaker')
    .agg(
        total_rows=('utterance', 'count'),
        rows_with_entities=('ent_count_clean', lambda x: (x > 0).sum()),
        total_entity_mentions=('ent_count_clean', 'sum'),
    )
    .reset_index()
)
speaker_stats['coverage_pct'] = (
    speaker_stats['rows_with_entities'] / speaker_stats['total_rows'] * 100
).round(2)
display(speaker_stats)

In [ ]:
# ── Section 8.8 : Entity length distribution ──────────────────────────────
display(HTML('<h3>8.8 · Entity Token-Length Distribution (clean)</h3>'))

clean_ent_df['token_count'] = clean_ent_df['entity_text'].str.split().apply(len)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, lbl in zip(axes, ['DISEASE', 'CHEMICAL']):
    sub = clean_ent_df[clean_ent_df['label'] == lbl]['token_count']
    sub.value_counts().sort_index().plot(
        kind='bar', ax=ax, color='steelblue' if lbl=='DISEASE' else 'tomato',
        edgecolor='black', linewidth=0.5
    )
    ax.set_title(f'{lbl} — token length distribution', fontsize=12)
    ax.set_xlabel('Number of tokens in entity')
    ax.set_ylabel('Frequency')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

In [ ]:
# ── Section 8.9 : Entities per utterance distribution ─────────────────────
display(HTML('<h3>8.9 · Entities-per-Utterance Distribution (clean)</h3>'))

fig, ax = plt.subplots(figsize=(10, 4))
counts = df['ent_count_clean'].value_counts().sort_index()
counts[counts.index <= 15].plot(kind='bar', ax=ax, color='mediumseagreen', edgecolor='black', linewidth=0.5)
ax.set_title('Number of clean entities per utterance', fontsize=12)
ax.set_xlabel('Entity count')
ax.set_ylabel('Number of utterances')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

print('\nDescriptive stats for entity count per utterance:')
display(df['ent_count_clean'].describe())

In [ ]:
# ── Section 8.10 : Label co-occurrence ────────────────────────────────────
display(HTML('<h3>8.10 · DISEASE / CHEMICAL Co-occurrence in Same Utterance</h3>'))

has_disease  = df['clean_labels'].apply(lambda l: 'DISEASE'  in l)
has_chemical = df['clean_labels'].apply(lambda l: 'CHEMICAL' in l)

cooccur = pd.DataFrame({
    'Category': [
        'Both DISEASE & CHEMICAL',
        'DISEASE only',
        'CHEMICAL only',
        'No entities',
    ],
    'Rows': [
        (has_disease & has_chemical).sum(),
        (has_disease & ~has_chemical).sum(),
        (~has_disease & has_chemical).sum(),
        (~has_disease & ~has_chemical).sum(),
    ]
})
cooccur['Pct'] = (cooccur['Rows'] / len(df) * 100).round(2)
display(cooccur)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(cooccur['Category'], cooccur['Rows'], color=['#4C72B0','#55A868','#C44E52','#8172B2'], edgecolor='black', linewidth=0.5)
ax.set_title('DISEASE / CHEMICAL co-occurrence per utterance', fontsize=12)
ax.set_ylabel('Number of utterances')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

---
## 9 · Validation Score Card

In [ ]:
display(HTML('<h2 style="color:#2a6496">✅ Validation Score Card</h2>'))

issues = []

if misaligned is not None and len(misaligned) > 0:
    issues.append(f'  ⚠ {len(misaligned):,} rows had misaligned entity/label counts (fixed by parsing)')
else:
    print('  ✓ All rows: entity count == label count')

if n_exact_dups > 0:
    issues.append(f'  ⚠ {n_exact_dups:,} exact duplicate utterances detected and removed')
else:
    print('  ✓ No exact duplicate utterances found')

n_unexpected = len(ent_df2[~ent_df2['label'].isin(VALID_LABELS)])
if n_unexpected > 0:
    issues.append(f'  ⚠ {n_unexpected:,} entities with unexpected labels found and removed')
else:
    print('  ✓ All entity labels are valid (DISEASE / CHEMICAL only)')

noisy_pct = n_noisy_mentions / n_raw_mentions * 100 if n_raw_mentions > 0 else 0
if noisy_pct > 1:
    issues.append(f'  ⚠ {noisy_pct:.2f}% of entity mentions were noisy/false-positives and removed')
else:
    print(f'  ✓ Low noise rate: {noisy_pct:.3f}% entity mentions removed')

if issues:
    print('\nIssues detected and resolved:')
    for issue in issues:
        print(issue)

print('\n--- Final Dataset Stats ---')
print(f'  Rows              : {len(df):,}')
print(f'  Total entities    : {df["ent_count_clean"].sum():,}')
print(f'  DISEASE mentions  : {(clean_ent_df["label"]=="DISEASE").sum():,}')
print(f'  CHEMICAL mentions : {(clean_ent_df["label"]=="CHEMICAL").sum():,}')
print(f'  Unique entities   : {clean_ent_df["entity_lower"].nunique():,}')

---
## 10 · Save Validated Output

In [ ]:
# ── Build final clean DataFrame ───────────────────────────────────────────
KEEP_COLS = [
    'dialogue_id', 'turn_id', 'speaker',
    'utterance', 'source_dataset', 'original_id',
    'extracted_entities_clean', 'entity_labels_clean',
    'ent_count_clean'
]

df_out = df[KEEP_COLS].copy()
df_out = df_out.rename(columns={
    'extracted_entities_clean': 'extracted_entities',
    'entity_labels_clean':      'entity_labels',
    'ent_count_clean':          'entity_count'
})

df_out.to_csv(OUTPUT_CSV, index=False)

print(f'Saved validated dataset to: {OUTPUT_CSV}')
print(f'Shape: {df_out.shape}')
df_out.head(3)

In [ ]:
# ── Quick sanity check on saved file ─────────────────────────────────────
check = pd.read_csv(OUTPUT_CSV, low_memory=False)
assert len(check) == len(df_out), 'Row count mismatch after save!'
assert list(check.columns) == list(df_out.columns), 'Column mismatch after save!'
print(f'Sanity check passed ✓')
print(f'  Rows    : {len(check):,}')
print(f'  Columns : {list(check.columns)}')